In [11]:
import pandas as pd
import os
from tqdm.auto import tqdm
from llm_asr_clarification.constants import SAMPLE_MEETINGS
import json
import torch
import torch.nn.functional as F
from rouge_score import rouge_scorer
from transformers import AutoTokenizer
from jiwer import wer
import jiwer

# Define a robust transformation pipeline
# This applies data cleaning steps in order, from top to bottom
JIWER_TRANSFORM = jiwer.Compose([
    jiwer.ToLowerCase(),                # Convert all text to lowercase
    jiwer.RemovePunctuation(),          # Strip characters like commas, periods, question marks
    jiwer.RemoveMultipleSpaces(),       # Turn multi-spaces into a single space
    jiwer.Strip(),                      # Clean up leading/trailing whitespaces
    jiwer.ReduceToListOfListOfWords()   # Format text tokens perfectly for jiwer's internal engine
])


pd.set_option('display.max_colwidth', None)

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"   # or another causal LM
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
scorer = rouge_scorer.RougeScorer(
    ["rougeL"],
    # use_stemmer=True,
    use_stemmer=False
)

def rouge_l(pred, ref):
    return scorer.score(
        ref,   # reference first
        pred   # prediction second
    )["rougeL"]

def load_df_from_path(AMI_PATH):
    meeting_paths = [entry.path for entry in os.scandir(AMI_PATH)]
        
    data = []
    for meeting_path in tqdm(meeting_paths):
        beam_results = os.path.join(meeting_path, "artifacts", f"beam_results.json")
        try:
            with open(beam_results, "r", encoding="utf-8") as f:
                lines = f.read()
            lines = json.loads(lines)
        except Exception as err:
            print(f"couldnt open file {beam_results}")
    
        for line in lines:
            for i in range(1,6):
                beam_no = f'beam_{i}'
                beam = line.pop(beam_no)
    
                line[f"{beam_no}_text"] = beam["text"]
                line[f"{beam_no}_asrlogprob"] = beam["asr_avg_log_prob"]
                line[f"{beam_no}_llmlogprob"] = beam["llm_avg_log_prob"]
                line[f"meeting_name"] = meeting_path.split("/")[-1]
                
    df = pd.DataFrame(lines)
    return df
    
AMI_TRAIN_PATH = '/group/jrwhitehill/amicorpus/train'
AMI_VAL_PATH = '/group/jrwhitehill/amicorpus/validation'

df = load_df_from_path(AMI_TRAIN_PATH)
df_val = load_df_from_path(AMI_VAL_PATH)

print(df.shape)

  0%|          | 0/136 [00:00<?, ?it/s]

  0%|          | 0/18 [00:00<?, ?it/s]

(1137, 17)


In [2]:
df.head(3)

,gt,beam_1_text,beam_1_asrlogprob,beam_1_llmlogprob,meeting_name,beam_2_text,beam_2_asrlogprob,beam_2_llmlogprob,beam_3_text,beam_3_asrlogprob,beam_3_llmlogprob,beam_4_text,beam_4_asrlogprob,beam_4_llmlogprob,beam_5_text,beam_5_asrlogprob,beam_5_llmlogprob
0,Hello.,Hello,-0.499981,-16.62500,TS3006c,Hello.,-0.166628,-10.25000,Hello.,-0.166628,-10.2500,Hello.,-0.166628,-10.2500,Hello!,-0.749961,-9.25000
1,..,I need to go there.,-0.825584,-8.00000,TS3006c,I think we are going to need to go.,-1.653785,-5.71875,But I'm gonna throw the ball away.,-1.753428,-5.8750,the,-64.277092,-25.6250,I'm gonna get back to the hospital.,-0.878668,-6.37500
2,"Yes, I made it.","Yes, I made it.",-0.060835,-5.09375,TS3006c,"Yes, I made it.",-0.060835,-5.09375,Yes I made it,-0.627244,-7.4375,"Yes, I made it",-0.239422,-5.6875,"Yes, I made it.",-0.060835,-5.09375


In [3]:
import re
import string

def normalize_text(text: str) -> str:
    """Mirrors the jiwer text normalization pipeline, except for last stage"""
    if not text:
        return ""
    
    # 1. jiwer.ToLowerCase()
    text = text.lower()
    
    # 2. jiwer.RemovePunctuation()
    # Removes standard punctuation: !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # 3. jiwer.RemoveMultipleSpaces()
    text = re.sub(r'\s+', ' ', text)
    
    # 4. jiwer.Strip()
    text = text.strip()
    
    return text

In [4]:
def process_columns(df):
    llm_logprob_columns = [f'beam_{i}_llmlogprob' for i in range(1,6)]
    asr_logprob_columns = [f'beam_{i}_asrlogprob' for i in range(1,6)]
    
    llm_logprobs = torch.tensor(df[llm_logprob_columns].values)
    asr_logprobs = torch.tensor(df[asr_logprob_columns].values)
    
    # # ALL BEAMS
    # scores = F.softmax(ALPHA*llm_logprobs + 0.0asr_logprobs, dim=1)
    # highest_score_idxs = torch.argmax(scores, dim=1, keepdim=True)
    # highest_scores = torch.gather(scores, dim=1, index=highest_score_idxs)

    # JUST BEAM 1
    scores = llm_logprobs
    highest_scores = scores[torch.arange(scores.size(0)), torch.zeros(scores.size(0), dtype=torch.long)]


    # AGGREGATED STATS
    df['max_llmlogprobs'] = torch.max(llm_logprobs, dim=1).values.numpy()
    df['min_llmlogprobs'] = torch.min(llm_logprobs, dim=1).values.numpy()
    df['spread_llmlogprobs'] = df['max_llmlogprobs'] - df['min_llmlogprobs']

    df['max_asrlogprobs'] = torch.max(asr_logprobs, dim=1).values.numpy()
    df['min_asrlogprobs'] = torch.min(asr_logprobs, dim=1).values.numpy()
    df['spread_asrlogprobs'] = df['max_asrlogprobs'] - df['min_asrlogprobs']

    df['highest_score'] = highest_scores.numpy()
    
    df['text'] = df['beam_1_text']
    df['num_tokens_text'] = df['beam_1_text'].apply(
        lambda x: len(tokenizer.encode(x))
    )
    df['num_tokens_gt'] = df['gt'].apply(
        lambda x: len(tokenizer.encode(x))
    )
    
    
    df["rougeL"] = [
        rouge_l(normalize_text(pred), normalize_text(ref)).fmeasure
        for pred, ref in zip(df["text"], df["gt"])
    ]
    df["wer"] = [
        min(1.0, wer(reference = ref, hypothesis = pred, reference_transform = JIWER_TRANSFORM, hypothesis_transform = JIWER_TRANSFORM))
        for pred, ref in zip(df["text"], df["gt"])
    ]

process_columns(df)
process_columns(df_val)

In [10]:
df.head(5)
print(df.shape)

(1137, 29)


# Find Fillers in the dataset

In [9]:
import pandas as pd
import re
from collections import Counter

def discover_fillers(df: pd.DataFrame, text_column: str, top_n: int = 100):
    """Finds the most common short utterances in your dataset."""
    short_utterances = []
    
    for text in df[text_column]:
        if not isinstance(text, str):
            continue
            
        # Clean the text
        clean_text = re.sub(r'[^\w\s]', '', text.lower()).strip()
        word_count = len(clean_text.split())
        
        # Look for 1 or 2 word utterances
        if 1 <= word_count <= 2:
            short_utterances.append(clean_text)
            
    # Return the most common ones
    return Counter(short_utterances).most_common(top_n)

# Example usage:
print(df.shape)
discovered_fillers = discover_fillers(df, 'text')
print(discovered_fillers)

(1137, 29)
[('you', 197), ('yeah', 42), ('okay', 11), ('thank you', 9), ('so', 7), ('the', 6), ('and', 5), ('all right', 5), ('yes', 4), ('yeah yeah', 4), ('thanks', 4), ('yeah okay', 3), ('thats it', 3), ('hello', 2), ('or', 2), ('woo', 2), ('right', 2), ('ah', 2), ('it', 2), ('one', 2), ('bye', 2), ('you know', 2), ('i think', 2), ('exactly', 2), ('and this', 2), ('and then', 2), ('one shape', 2), ('oh', 2), ('im sorry', 2), ('thats', 2), ('and with', 2), ('yes okay', 2), ('good', 2), ('hmm', 1), ('just kidding', 1), ('i see', 1), ('yeah conceptual', 1), ('in the', 1), ('probably', 1), ('welcome', 1), ('go yes', 1), ('good job', 1), ('so i', 1), ('but this', 1), ('but just', 1), ('we dont', 1), ('my', 1), ('yeah also', 1), ('year in', 1), ('now yeah', 1), ('materials', 1), ('for the', 1), ('but then', 1), ('interesting electronics', 1), ('the ability', 1), ('up', 1), ('single curved', 1), ('everything direct', 1), ('every direct', 1), ('presence', 1), ('to these', 1), ('for everythin

# Stop Word Removal Analysis

In [ ]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords

# Download standard NLTK stopwords if you haven't already
nltk.download('stopwords', quiet=True)

# 1. Define your complete set of stop words
standard_stops = set(stopwords.words('english'))

# Add ASR-specific conversational fillers (no punctuation, all lowercase)
asr_fillers = {
    "yeah", "alright", "good", "mmhmm", "mm", "hmm", 
    "uh", "um", "ah", "okay", "ok", "oh"
}
all_stop_words = standard_stops.union(asr_fillers)

def contains_meaningful_text(text: str) -> bool:
    """
    Returns True if the text contains non-stop words after cleaning.
    """
    if not isinstance(text, str) or not text.strip():
        return False
        
    # Lowercase the text
    text = text.lower()
    
    # Remove punctuation. 
    # Note: "Mm-hmm" becomes "mmhmm", which is why "mmhmm" is in our custom list above.
    text = re.sub(r'[^\w\s]', '', text)
    
    # Split into words and check against stop words
    words = text.split()
    meaningful_words = [w for w in words if w not in all_stop_words]
    
    # If the list is not empty, there is meaningful text left
    return len(meaningful_words) > 0

# --- Example Usage ---

# Sample DataFrame based on your description
data = {
    'generated text': ["Yeah yeah yeah", "Hello world", "Alright, good", "The model works"],
    'ground truth text': ["yeah", "hi world", "mm-hmm", "model works fine"],
    'ROUGE-L': [0.5, 0.8, 0.0, 0.9],
    'WER': [0.2, 0.3, 1.0, 0.1]
}
test_df = pd.DataFrame(data)

# 2. Create boolean masks for both text columns
mask_gen = test_df['generated text'].apply(contains_meaningful_text)
mask_gt = test_df['ground truth text'].apply(contains_meaningful_text)

# 3. Filter the DataFrame to keep rows where BOTH texts are meaningful
# (If you want to drop rows where AT LEAST ONE is empty, use the `&` operator)
df_filtered = test_df[mask_gen & mask_gt].copy()

print("Original DataFrame:")
print(test_df)
print("\nFiltered DataFrame:")
print(df_filtered)